# Phase 3: Feature Engineering
In this phase, we aim to extract new predictive variables from the raw dataset. [cite_start]According to our project outline, we must rely only on observable physical characteristics to predict a building's energy performance[cite: 26]. 

Our main tasks are:
1. **Deduplication:** Removing multiple EPC records for the same dwelling to prevent data leakage[cite: 35].
2. **Feature Creation:** Calculating `feature_energy_density` and `feature_building_age` proxies.

*Note: Data transformation steps that learn from the data (like scaling) will be strictly handled inside a Cross-Validation Pipeline during Phase 4 to avoid data leakage[cite: 53].*

In [46]:
import pandas as pd
import numpy as np

# Load the raw dataset
# We load the raw dataset directly to preserve columns like BUILDING_REFERENCE_NUMBER and INSPECTION_DATE
df_raw = pd.read_parquet('manchester_epc_prepped.parquet')

print(f"Initial raw dataset size: {len(df_raw)} rows")

Initial raw dataset size: 334270 rows


## 1. Deduplication (Leakage Prevention)
To avoid target leakage caused by updated certificates for the same property, we inspect the dataset for duplicate entries and retain only the most recent record per dwelling[cite: 35].

In [47]:
# Convert inspection date to datetime objects for accurate sorting
df_raw['INSPECTION_DATE'] = pd.to_datetime(df_raw['INSPECTION_DATE'])

# Sort by inspection date and keep the latest certificate for each unique building
df_feat = df_raw.sort_values('INSPECTION_DATE').drop_duplicates(subset=['BUILDING_REFERENCE_NUMBER'], keep='last').copy()

print(f"Dataset size after dropping duplicates: {len(df_feat)} rows")
# We expect this to drop the dataset size to approximately 150,000 records.

Dataset size after dropping duplicates: 266470 rows


# 2. Add flag features


In [48]:
is_pre1919 = df_feat['construction_year_estimated'] < 1919

if 'wall_type_cavity' in df_feat.columns:
    df_feat['pre1919_renovated_cavity_wall'] = (is_pre1919 & (df_feat['wall_type_cavity'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_cavity_wall'] = 0

if 'wall_insulation_external' in df_feat.columns:
    df_feat['pre1919_renovated_ext_insulation'] = (is_pre1919 & (df_feat['wall_insulation_external'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_ext_insulation'] = 0

if 'wall_insulation_internal' in df_feat.columns:
    df_feat['pre1919_renovated_int_insulation'] = (is_pre1919 & (df_feat['wall_insulation_internal'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_int_insulation'] = 0

if 'wall_insulation_filled_cavity' in df_feat.columns:
    df_feat['pre1919_renovated_filled_cavity'] = (is_pre1919 & (df_feat['wall_insulation_filled_cavity'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_filled_cavity'] = 0

if 'win_glazing_double' in df_feat.columns:
    df_feat['pre1919_renovated_double_glazing'] = (is_pre1919 & (df_feat['win_glazing_double'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_double_glazing'] = 0

if 'win_glazing_triple' in df_feat.columns:
    df_feat['pre1919_renovated_triple_glazing'] = (is_pre1919 & (df_feat['win_glazing_triple'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_triple_glazing'] = 0

if 'win_glazing_high_perf' in df_feat.columns:
    df_feat['pre1919_renovated_high_perf_glazing'] = (is_pre1919 & (df_feat['win_glazing_high_perf'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_high_perf_glazing'] = 0

if 'win_extent_full' in df_feat.columns:
    df_feat['pre1919_renovated_full_windows'] = (is_pre1919 & (df_feat['win_extent_full'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_full_windows'] = 0

# 🔥 Heating 
if 'heat_sys_heat_pump' in df_feat.columns:
    df_feat['pre1919_renovated_heat_pump'] = (is_pre1919 & (df_feat['heat_sys_heat_pump'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_heat_pump'] = 0

if 'heat_sys_underfloor' in df_feat.columns:
    df_feat['pre1919_renovated_underfloor'] = (is_pre1919 & (df_feat['heat_sys_underfloor'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_underfloor'] = 0

if 'heat_energy_electric' in df_feat.columns:
    df_feat['pre1919_renovated_electric'] = (is_pre1919 & (df_feat['heat_energy_electric'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_electric'] = 0

# 🏠 Roof 
if 'roof_insulation_thickness_mm' in df_feat.columns:
    df_feat['pre1919_renovated_thick_roof'] = (is_pre1919 & (df_feat['roof_insulation_thickness_mm'] > 100)).astype(int)
else:
    df_feat['pre1919_renovated_thick_roof'] = 0

if 'roof_insulated_generic' in df_feat.columns:
    df_feat['pre1919_renovated_roof_insulated'] = (is_pre1919 & (df_feat['roof_insulated_generic'] == 1)).astype(int)
else:
    df_feat['pre1919_renovated_roof_insulated'] = 0

# 확인
flag_cols = [c for c in df_feat.columns if c.startswith('pre1919_')]
print("=== Pre-1919 Renovation Flag Counts ===")
for col in flag_cols:
    counts = df_feat[col].value_counts().sort_index()
    total = len(df_feat)
    pct = counts.get(1, 0) / total * 100
    print(f"{col}: 0 → {counts.get(0, 0):,} | 1 → {counts.get(1, 0):,} ({pct:.1f}%)")

drop_flags = [
    'pre1919_renovated_ext_insulation',
    'pre1919_renovated_int_insulation',
    'pre1919_renovated_triple_glazing',
    'pre1919_renovated_high_perf_glazing',
    'pre1919_renovated_heat_pump',
    'pre1919_renovated_underfloor',
    'pre1919_renovated_roof_insulated',
    'pre1919_renovated_filled_cavity',
   'pre1919_renovated_electric',
]

df_feat.drop(columns=[c for c in drop_flags if c in df_feat.columns], inplace=True)

keep_flags = [
    'pre1919_renovated_cavity_wall',
    'pre1919_renovated_double_glazing',
    'pre1919_renovated_full_windows',
    'pre1919_renovated_thick_roof',
]

print("=== Kept flags ===")
for col in keep_flags:
    counts = df_feat[col].value_counts().sort_index()
    pct = counts.get(1, 0) / len(df_feat) * 100
    print(f"{col}: 0 → {counts.get(0, 0):,} | 1 → {counts.get(1, 0):,} ({pct:.1f}%)")

=== Pre-1919 Renovation Flag Counts ===
pre1919_renovated_cavity_wall: 0 → 242,596 | 1 → 23,874 (9.0%)
pre1919_renovated_ext_insulation: 0 → 265,457 | 1 → 1,013 (0.4%)
pre1919_renovated_int_insulation: 0 → 263,966 | 1 → 2,504 (0.9%)
pre1919_renovated_filled_cavity: 0 → 258,873 | 1 → 7,597 (2.9%)
pre1919_renovated_double_glazing: 0 → 208,855 | 1 → 57,615 (21.6%)
pre1919_renovated_triple_glazing: 0 → 266,408 | 1 → 62 (0.0%)
pre1919_renovated_high_perf_glazing: 0 → 266,169 | 1 → 301 (0.1%)
pre1919_renovated_full_windows: 0 → 216,096 | 1 → 50,374 (18.9%)
pre1919_renovated_heat_pump: 0 → 266,379 | 1 → 91 (0.0%)
pre1919_renovated_underfloor: 0 → 266,305 | 1 → 165 (0.1%)
pre1919_renovated_electric: 0 → 260,784 | 1 → 5,686 (2.1%)
pre1919_renovated_thick_roof: 0 → 230,206 | 1 → 36,264 (13.6%)
pre1919_renovated_roof_insulated: 0 → 264,489 | 1 → 1,981 (0.7%)
=== Kept flags ===
pre1919_renovated_cavity_wall: 0 → 242,596 | 1 → 23,874 (9.0%)
pre1919_renovated_double_glazing: 0 → 208,855 | 1 → 57,615

## 3. Final Cleanup & Export (Preventing Data Leakage)
To build a robust model that strictly relies on physical building characteristics, we must remove identification columns and variables that cause target leakage.

Specifically, we drop:
* `ENERGY_CONSUMPTION_CURRENT` — outcome-related, leaky as a feature.
* `CURRENT_ENERGY_RATING` — already converted to `target_is_efficient`.
* ID and Date columns.

**`CURRENT_ENERGY_EFFICIENCY` is kept** as the regression target (`y`), per professor feedback. `target_is_efficient` is kept for post-prediction binning (predicted score ≥ 69 → efficient).

In [49]:
# Drop columns that cause leakage or are just non-predictive IDs
cols_to_drop_final = [
    'BUILDING_REFERENCE_NUMBER', 
    'INSPECTION_DATE', 
    'CURRENT_ENERGY_RATING',       # Replaced by target_is_efficient
    'ENERGY_CONSUMPTION_CURRENT',  # Outcome-related — leaky as a feature
    # CURRENT_ENERGY_EFFICIENCY is kept as the regression target (y)
    # target_is_efficient is kept for post-prediction binning/evaluation
]

df_final = df_feat.drop(columns=[col for col in cols_to_drop_final if col in df_feat.columns])

print("=== PHASE 3 FINAL DATASET ===")
print(f"Final shape ready for Phase 4: {df_final.shape}")
print("\nTarget columns confirmed present:")
print(f"  - CURRENT_ENERGY_EFFICIENCY : min={df_final['CURRENT_ENERGY_EFFICIENCY'].min()}, max={df_final['CURRENT_ENERGY_EFFICIENCY'].max()}")
print(f"  - target_is_efficient counts: {df_final['target_is_efficient'].value_counts().to_dict()}")

output_filename = 'manchester_epc_phase3_final.parquet'
df_final.to_parquet(output_filename)
print(f"\nSUCCESS: Phase 3 is complete! Clean and engineered data saved as '{output_filename}'.")

=== PHASE 3 FINAL DATASET ===
Final shape ready for Phase 4: (266470, 73)

Target columns confirmed present:
  - CURRENT_ENERGY_EFFICIENCY : min=1, max=147
  - target_is_efficient counts: {1: 155584, 0: 110886}

SUCCESS: Phase 3 is complete! Clean and engineered data saved as 'manchester_epc_phase3_final.parquet'.
